# FT -> XGB/ResNet Prediction (Template)

Load trained FT and XGB/ResNet models to predict on new data.

**Usage**:
1. Copy this notebook to your working directory
2. Train models using Train_FT_Embed_XGBResN.ipynb first
3. Modify the "Configuration" section below
4. Run all cells

**What it does**:
1. Load FT model and generate embeddings for new data
2. Load XGB/ResNet model and predict using embeddings
3. Save predictions to CSV

In [ ]:
from pathlib import Path
import os

# Work in current directory - compatible with pip-installed package
work_dir = Path.cwd()

os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import json
import joblib
import pandas as pd
import torch

from ins_pricing.production.predict import load_predictor_from_config

In [ ]:
# ============================================================
# CONFIGURATION - Modify these paths for your setup
# ============================================================

# Config files from training (generated by Train_FT_Embed_XGBResN.ipynb)
ft_cfg_path = work_dir / 'config_ft_unsupervised_ddp_embed.json'
xgb_cfg_path = work_dir / 'config_xgb_from_ft_unsupervised.json'
resn_cfg_path = work_dir / 'config_resn_from_ft_unsupervised_ddp.json'

# New data to predict (must have same features as training data)
input_path = work_dir / 'Data/od_bc_new.csv'

# Output file for predictions
output_path = work_dir / 'Results/predictions_ft_xgb.csv'

# Model name (leave as None to auto-detect from config)
model_name = None

# Which models to use for prediction: ["xgb"], ["resn"], or ["xgb", "resn"]
model_keys = ["xgb", "resn"]

print(f"Working directory: {work_dir}")
print(f"Input data: {input_path}")
print(f"Output file: {output_path}")
print(f"Models to use: {model_keys}")

In [ ]:
# ============================================================
# LOAD CONFIGURATIONS (No changes needed below)
# ============================================================

print("Loading configurations...")
ft_cfg = json.loads(ft_cfg_path.read_text(encoding='utf-8'))
xgb_cfg = (
    json.loads(xgb_cfg_path.read_text(encoding='utf-8'))
    if "xgb" in model_keys
    else None
)
resn_cfg = (
    json.loads(resn_cfg_path.read_text(encoding='utf-8'))
    if "resn" in model_keys
    else None
)

if model_name is None:
    model_list = list(ft_cfg.get('model_list') or [])
    model_categories = list(ft_cfg.get('model_categories') or [])
    if len(model_list) != 1 or len(model_categories) != 1:
        raise ValueError('Set model_name in config section when multiple models exist.')
    model_name = f"{model_list[0]}_{model_categories[0]}"

ft_output_dir = (ft_cfg_path.parent / ft_cfg['output_dir']).resolve()
xgb_output_dir = (
    (xgb_cfg_path.parent / xgb_cfg['output_dir']).resolve()
    if xgb_cfg is not None
    else None
)
ft_prefix = ft_cfg.get('ft_feature_prefix', 'ft_emb')
xgb_task_type = str(xgb_cfg.get('task_type', 'regression')) if xgb_cfg else None

if ft_cfg.get('geo_feature_nmes'):
    raise ValueError('FT with geo tokens not supported in this template.')

print(f"✓ Model name: {model_name}")
print(f"✓ FT output: {ft_output_dir}")
if xgb_output_dir:
    print(f"✓ XGB output: {xgb_output_dir}")

In [ ]:
# ============================================================
# STEP 1: Generate embeddings with FT model
# ============================================================

print("\n[Step 1/3] Loading FT model and generating embeddings...")
ft_model_path = ft_output_dir / 'model' / f"01_{model_name}_FTTransformer.pth"
print(f"FT model: {ft_model_path}")

ft_payload = torch.load(ft_model_path, map_location='cpu')
if isinstance(ft_payload, dict) and 'model' in ft_payload:
    ft_model = ft_payload['model']
else:
    ft_model = ft_payload

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if hasattr(ft_model, 'device'):
    ft_model.device = device
if hasattr(ft_model, 'to'):
    ft_model.to(device)
if hasattr(ft_model, 'ft'):
    ft_model.ft.to(device)

print(f"\nLoading input data...")
df_new = pd.read_csv(input_path)
print(f"✓ Loaded {len(df_new):,} rows")

print(f"Generating embeddings...")
emb = ft_model.predict(df_new, return_embedding=True)
emb_cols = [f"pred_{ft_prefix}_{i}" for i in range(emb.shape[1])]
df_with_emb = df_new.copy()
df_with_emb[emb_cols] = emb
print(f"✓ Generated {len(emb_cols)} embedding features")

result = df_with_emb.copy()

In [ ]:
# ============================================================
# STEP 2: Predict with XGB model
# ============================================================

if "xgb" in model_keys:
    print("\n[Step 2/3] Loading XGB model and predicting...")
    xgb_model_path = xgb_output_dir / 'model' / f"01_{model_name}_Xgboost.pkl"
    print(f"XGB model: {xgb_model_path}")
    
    xgb_payload = joblib.load(xgb_model_path)
    if isinstance(xgb_payload, dict) and 'model' in xgb_payload:
        xgb_model = xgb_payload['model']
        feature_list = (
            xgb_payload.get('preprocess_artifacts', {})
            .get('factor_nmes')
        )
    else:
        xgb_model = xgb_payload
        feature_list = None

    if not feature_list:
        feature_list = xgb_cfg.get('feature_list') or []
    if not feature_list:
        raise ValueError('Feature list missing in model.')

    print(f"Using {len(feature_list)} features")
    X = df_with_emb[feature_list]
    
    if xgb_task_type == 'classification' and hasattr(xgb_model, 'predict_proba'):
        pred = xgb_model.predict_proba(X)[:, 1]
    else:
        pred = xgb_model.predict(X)

    result['pred_xgb'] = pred
    print(f"✓ XGB predictions: min={pred.min():.4f}, max={pred.max():.4f}, mean={pred.mean():.4f}")

# ============================================================
# STEP 3: Predict with ResNet model
# ============================================================

if "resn" in model_keys:
    print("\n[Step 3/3] Loading ResNet model and predicting...")
    resn_predictor = load_predictor_from_config(
        resn_cfg_path,
        "resn",
        model_name=model_name,
    )
    pred_resn = resn_predictor.predict(df_with_emb)
    result['pred_resn'] = pred_resn
    print(f"✓ ResNet predictions: min={pred_resn.min():.4f}, max={pred_resn.max():.4f}, mean={pred_resn.mean():.4f}")

# ============================================================
# SAVE RESULTS
# ============================================================

print(f"\nSaving predictions to: {output_path}")
output_path.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output_path, index=False)
print(f"✓ Saved {len(result):,} rows")

print("\n" + "="*60)
print("PREDICTION COMPLETE")
print("="*60)
print(f"Output file: {output_path}")
print(f"Columns: {', '.join([c for c in result.columns if c.startswith('pred_')])}")
print("\nFirst 5 predictions:")
display(result[[c for c in result.columns if c.startswith('pred_')]].head())